# Cross-Domain Material Discovery

Systematic sweep across battery-polymer, battery-metal, and ceramic-metal boundaries.
Finds surprising high/low scorers and ranks multi-domain designs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import json

csv_path = '../showcase/outputs/cross_domain_surprises.csv'
json_path = '../showcase/outputs/multi_domain_designs.json'

if not os.path.exists(csv_path):
    print('Generating data...')
    from showcase.cross_domain_discovery import main
    main()

df = pd.read_csv(csv_path)
with open(json_path) as f:
    data = json.load(f)

print(f'Total cross-domain pairs: {data["total_cross_pairs"]}')
print(f'Multi-domain designs: {len(data["multi_domain_designs"])}')
print()
print('Pairs by functor:')
print(df['functor'].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Score distribution by functor
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, functor in zip(axes, ['battery_polymer', 'battery_metal', 'ceramic_metal']):
    subset = df[df['functor'] == functor]['score'].dropna()
    ax.hist(subset, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(0.5, color='red', linestyle='--', label='Viability threshold')
    ax.set_title(functor.replace('_', '-').title())
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.suptitle('Cross-Domain Score Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Surprise table: top 10
print('TOP 10: Highest-Scoring Cross-Domain Pairs')
print('=' * 70)
for r in data['top_10_surprises']:
    compat = 'OK' if r['compatible'] else 'CAUTION'
    print(f"  {r['material_a']:12s} + {r['material_b']:20s} "
          f"({r['functor']:15s}) = {r['score']:.4f} [{compat}]")

print()
print('BOTTOM 10: Lowest-Scoring Cross-Domain Pairs')
print('=' * 70)
for r in data['bottom_10_warnings']:
    compat = 'OK' if r['compatible'] else 'FAIL'
    print(f"  {r['material_a']:12s} + {r['material_b']:20s} "
          f"({r['functor']:15s}) = {r['score']:.4f} [{compat}]")

In [ ]:
# Battery-Polymer heatmap
bp = df[df['functor'] == 'battery_polymer'].copy()
if not bp.empty:
    pivot = bp.pivot(index='material_a', columns='material_b', values='score')

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('Battery-Polymer Cross-Domain Scores')
    ax.set_xlabel('Battery Material')
    ax.set_ylabel('Polymer')
    plt.colorbar(im, label='Score')

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

    plt.tight_layout()
    plt.show()

In [ ]:
# Multi-domain design rankings
designs = data['multi_domain_designs']

fig, ax = plt.subplots(figsize=(10, 5))
valid_designs = [d for d in designs if 'error' not in d]
names = [d['name'].split(':')[1].strip() if ':' in d['name'] else d['name']
         for d in valid_designs]
scores = [d['overall_score'] for d in valid_designs]
colors = ['#4caf50' if d['viable'] else '#d32f2f' for d in valid_designs]

bars = ax.barh(names, scores, color=colors, edgecolor='black', alpha=0.8)
ax.axvline(0.5, color='orange', linestyle='--', linewidth=2, label='Viability threshold')
ax.set_xlabel('Overall Score')
ax.set_title('Multi-Domain Design Rankings')
ax.legend()

for bar, score, design in zip(bars, scores, valid_designs):
    bn = design.get('bottleneck', '')
    label = f'{score:.3f}'
    if bn:
        label += f' (bottleneck: {bn.split("(")[0].strip()})'
    ax.text(max(score + 0.01, 0.02), bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=7)

plt.tight_layout()
plt.show()